In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
 
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [2]:
PROCESSED_DIR = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/processed_data"
MODEL_DIR     = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/model/models"
 
# Thresholds to test — 100 values between 0.05 and 0.95
THRESHOLDS = np.linspace(0.05, 0.95, 100)
 
BATCH_SIZE   = 64      # can be larger here since we are not backpropagating
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
TUNE_GROUPS = [
    "reproductive_process",
    "interspecies_interaction",
    "immune_system_process",
    "molecular_transducer",
    "mf_regulator_activity",
    "homeostatic_process",
    "atp_dependent_activity",
    "oxidoreductase_activity",
    "transferase_activity",
    "hydrolase_activity",
    "lyase_activity",
    "ion_transport",
    "vesicle_mediated_transport",
    "protein_transport",
    "lipid_transport",
    "nuclear_transport",
    "protein_binding",
    "dna_binding",
    "rna_binding",
    "lipid_binding",
    "metal_ion_binding",
    "small_molecule_binding",
]
 
print(f"Device : {DEVICE}")
print(f"Testing {len(THRESHOLDS)} thresholds per group")
print(f"Groups : {len(TUNE_GROUPS)}")
 

Device : cpu
Testing 100 thresholds per group
Groups : 22


In [3]:
class ProteinMLP(nn.Module):
    def __init__(self, input_dim=428, num_classes=128, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
 
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
 
            nn.Linear(256, num_classes),
        )
 
    def forward(self, x):
        return self.network(x)
 
print("✓ ProteinMLP defined")

✓ ProteinMLP defined


In [4]:
class ProteinDataset(torch.utils.data.Dataset):
    def __init__(self, indices, feature_matrix, label_matrix):
        self.indices        = indices
        self.feature_matrix = feature_matrix
        self.label_matrix   = label_matrix
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, idx):
        i        = self.indices[idx]
        features = torch.tensor(self.feature_matrix[i], dtype=torch.float32)
        label    = torch.tensor(self.label_matrix[i],   dtype=torch.float32)
        return features, label
 
print("✓ Dataset class defined")

✓ Dataset class defined


In [7]:
def get_val_probabilities(model, val_loader, device):
    """
    Run the model on the validation set and return raw sigmoid probabilities.
    We collect all probabilities first, then test thresholds — this is much
    faster than running the model once per threshold value.
    """
    model.eval()
    all_probs   = []
    all_targets = []
 
    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            logits   = model(features)
            probs    = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            all_targets.append(labels.numpy())
 
    return np.vstack(all_probs), np.vstack(all_targets)
 
 
def find_best_threshold(probs, targets, thresholds):
    """
    Test each threshold value and return the one with the highest macro F1.
    Also returns per-threshold metrics for plotting if needed.
    """
    results = []
 
    for thresh in thresholds:
        preds     = (probs >= thresh).astype(float)
        f1        = f1_score(targets,  preds, average="macro", zero_division=0)
        precision = precision_score(targets, preds, average="macro", zero_division=0)
        recall    = recall_score(targets,    preds, average="macro", zero_division=0)
        results.append({
            "threshold": round(float(thresh), 4),
            "f1":        round(float(f1), 6),
            "precision": round(float(precision), 6),
            "recall":    round(float(recall), 6),
        })
 
    # Sort by F1 descending, pick best
    results.sort(key=lambda x: -x["f1"])
    return results[0], results    # best result, all results
 
 
def tune_group(group_name, processed_dir, model_dir, thresholds, device, batch_size):
    """
    Full tuning pipeline for one group:
      1. Load processed data (features, labels, go_terms)
      2. Recreate the exact 80/10/10 split from training (random_state=42)
         and verify it matches the saved test_idx.json
      3. Load the best saved model
      4. Get validation set probabilities
      5. Find the threshold that maximises macro F1 on the validation set
    """
    group_dir  = os.path.join(processed_dir, group_name)
    model_path = os.path.join(model_dir, f"{group_name}_best.pt")
 
    # ── Check files exist ──────────────────────────────────────────────────
    if not os.path.exists(model_path):
        print(f"  ✗ {group_name}: model file not found at {model_path}")
        return None
 
    # ── Load data ──────────────────────────────────────────────────────────
    with open(os.path.join(group_dir, "accessions.json")) as f:
        accessions = json.load(f)
 
    feature_matrix = np.load(os.path.join(group_dir, "features.npy"))
    label_matrix   = np.load(os.path.join(group_dir, "labels.npy"))
 
    with open(os.path.join(group_dir, "go_terms.json")) as f:
        go_term_list = json.load(f)
 
    n_classes = len(go_term_list)
 
    # ── Recreate the exact same train/val/test split from training ─────────
    # Must exactly match build_dataloaders() in train_model.ipynb
    indices = list(range(len(accessions)))
    train_val_idx, test_idx = train_test_split(
        indices, test_size=0.1, random_state=42
    )
    _, val_idx = train_test_split(
        train_val_idx, test_size=0.111, random_state=42
    )

    # Verify against saved test_idx to catch any split mismatch
    test_idx_path = os.path.join(group_dir, "test_idx.json")
    if os.path.exists(test_idx_path):
        with open(test_idx_path) as f:
            saved_test_idx = set(json.load(f))
        if set(test_idx) != saved_test_idx:
            raise RuntimeError(
                f"{group_name}: computed test_idx does not match saved test_idx.json. "
                "The split in tune_group() does not match build_dataloaders(). "
                "Fix the split before tuning."
            )

    val_dataset = ProteinDataset(val_idx, feature_matrix, label_matrix)
    val_loader  = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
 
    # ── Load model ─────────────────────────────────────────────────────────
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model      = ProteinMLP(num_classes=n_classes).to(device)
    model.load_state_dict(checkpoint["model_state"])
 
    trained_f1 = checkpoint.get("val_f1", None)
 
    # ── Get probabilities ──────────────────────────────────────────────────
    probs, targets = get_val_probabilities(model, val_loader, device)
 
    # ── Find best threshold ────────────────────────────────────────────────
    best, all_results = find_best_threshold(probs, targets, thresholds)
 
    # ── Compute metrics at default threshold 0.5 for comparison ───────────
    default_preds = (probs >= 0.5).astype(float)
    default_f1    = f1_score(targets, default_preds, average="macro", zero_division=0)
    default_p     = precision_score(targets, default_preds, average="macro", zero_division=0)
    default_r     = recall_score(targets, default_preds, average="macro", zero_division=0)
 
    return {
        "group_name":      group_name,
        "n_go_terms":      n_classes,
        "n_val_proteins":  len(val_idx),
        "trained_val_f1":  round(float(trained_f1), 6) if trained_f1 else None,
        "default_threshold": {
            "threshold": 0.5,
            "f1":        round(float(default_f1), 6),
            "precision": round(float(default_p), 6),
            "recall":    round(float(default_r), 6),
        },
        "optimal_threshold": best,
        "f1_gain":           round(best["f1"] - float(default_f1), 6),
        "all_thresholds":    all_results,   # full curve saved for plotting
    }
 
print("✓ Tuning functions defined")

✓ Tuning functions defined


In [8]:
print("\nRunning threshold tuning...")
print("=" * 70)
 
all_results = {}
 
for group_name in TUNE_GROUPS:
    print(f"\n  {group_name}")
    result = tune_group(
        group_name   = group_name,
        processed_dir = PROCESSED_DIR,
        model_dir    = MODEL_DIR,
        thresholds   = THRESHOLDS,
        device       = DEVICE,
        batch_size   = BATCH_SIZE,
    )
 
    if result is None:
        continue
 
    all_results[group_name] = result
 
    d = result["default_threshold"]
    o = result["optimal_threshold"]
 
    print(f"    Val proteins  : {result['n_val_proteins']:,}")
    print(f"    GO terms      : {result['n_go_terms']}")
    print(f"    Default (0.50): F1={d['f1']:.4f}  P={d['precision']:.4f}  R={d['recall']:.4f}")
    print(f"    Optimal ({o['threshold']:.2f}): F1={o['f1']:.4f}  P={o['precision']:.4f}  R={o['recall']:.4f}")
    print(f"    F1 gain       : +{result['f1_gain']:.4f}")


Running threshold tuning...

  reproductive_process
    Val proteins  : 627
    GO terms      : 72
    Default (0.50): F1=0.2154  P=0.1418  R=0.6338
    Optimal (0.88): F1=0.2795  P=0.3097  R=0.3450
    F1 gain       : +0.0642

  interspecies_interaction
    Val proteins  : 892
    GO terms      : 111
    Default (0.50): F1=0.3952  P=0.2943  R=0.7822
    Optimal (0.87): F1=0.5049  P=0.5123  R=0.5911
    F1 gain       : +0.1097

  immune_system_process
    Val proteins  : 505
    GO terms      : 106
    Default (0.50): F1=0.3831  P=0.2787  R=0.7340
    Optimal (0.82): F1=0.4782  P=0.5082  R=0.5177
    F1 gain       : +0.0951

  molecular_transducer
    Val proteins  : 305
    GO terms      : 50
    Default (0.50): F1=0.6246  P=0.5090  R=0.9305
    Optimal (0.92): F1=0.7615  P=0.7828  R=0.7955
    F1 gain       : +0.1370

  mf_regulator_activity
    Val proteins  : 1,224
    GO terms      : 98
    Default (0.50): F1=0.4259  P=0.3248  R=0.8115
    Optimal (0.88): F1=0.5672  P=0.5897  R=0

In [ ]:
print(f"\n{'='*70}")
print("  THRESHOLD TUNING SUMMARY")
print(f"{'='*70}")
print(f"  {'Group':<35} {'Old F1':>7}  {'New F1':>7}  {'Threshold':>9}  {'Gain':>7}")
print(f"  {'-'*34} {'-'*7}  {'-'*7}  {'-'*9}  {'-'*7}")
 
for group_name, result in all_results.items():
    d = result["default_threshold"]
    o = result["optimal_threshold"]
    print(
        f"  {group_name:<35} {d['f1']:>7.4f}  {o['f1']:>7.4f}  "
        f"{o['threshold']:>9.2f}  +{result['f1_gain']:>6.4f}"
    )
 
# Save results — the inference pipeline will read optimal thresholds from here
output_path = os.path.join(MODEL_DIR, "threshold_results.json")
# Remove all_thresholds from saved file to keep it readable
save_results = {}
for group_name, result in all_results.items():
    save_results[group_name] = {k: v for k, v in result.items() if k != "all_thresholds"}
 
with open(output_path, "w") as f:
    json.dump(save_results, f, indent=2)
 
print(f"\n✓ Results saved to {output_path}")
print("  The inference pipeline will automatically use the optimal thresholds.")
print("  Next step: build the inference pipeline.")


  THRESHOLD TUNING SUMMARY
  Group                                Old F1   New F1  Threshold     Gain
  ---------------------------------- -------  -------  ---------  -------
  reproductive_process                 0.2154   0.2795       0.88  +0.0642
  interspecies_interaction             0.3952   0.5049       0.87  +0.1097
  immune_system_process                0.3831   0.4782       0.82  +0.0951
  molecular_transducer                 0.6246   0.7615       0.92  +0.1370
  mf_regulator_activity                0.4259   0.5672       0.88  +0.1414
  homeostatic_process                  0.3995   0.5450       0.89  +0.1456
  atp_dependent_activity               0.5822   0.7069       0.93  +0.1247
  oxidoreductase_activity              0.3553   0.5812       0.95  +0.2259
  transferase_activity                 0.3558   0.5546       0.93  +0.1988
  hydrolase_activity                   0.3062   0.5045       0.93  +0.1983
  lyase_activity                       0.5818   0.6927       0.89  +0.110